# RPA — Extração de clientes + envio ao N8N

Este notebook é o "robô" do projeto: ele **lê a tabela de clientes** de uma página web (BeautifulSoup) e **envia os dados ao Webhook do N8N**, que cruza o perfil de cada cliente com as opções de investimento e devolve uma mensagem personalizada.

| Etapa | O que acontece |
|---|---|
| 1 | Baixar a página de clientes (GitHub Pages) |
| 2 | Extrair a tabela `#clientes` |
| 3 | Enviar `{"clientes": [...]}` por POST ao Webhook do N8N |
| 4 | Mostrar as mensagens devolvidas |

> Versão em script (linha de comando): `src/extrair_clientes.py`.

## 1. Dependências

In [ ]:
!pip install -q requests beautifulsoup4

## 2. Configuração

Cole abaixo a URL do Webhook do N8N (nó **Webhook → Production URL**, com o workflow **ativo**).
Para testar sem ativar o workflow, use a **Test URL** (`/webhook-test/...`) e clique em *Listen for test event* no N8N antes de executar.

In [ ]:
URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

# Exemplo: "https://SEU-USUARIO.app.n8n.cloud/webhook/clientes"
N8N_WEBHOOK = ""   # deixe vazio para apenas extrair (modo dry-run)

## 3. Extração (RPA)

In [ ]:
import requests
from bs4 import BeautifulSoup

CAMPOS = ("nome", "email", "saldo", "perfil")

# 1) Baixar a página
resposta = requests.get(URL_PAGINA, timeout=15)
resposta.raise_for_status()
resposta.encoding = "utf-8"

# 2) Ler a tabela de clientes
soup = BeautifulSoup(resposta.text, "html.parser")
clientes = []
for linha in soup.select("#clientes tbody tr"):
    cliente = {}
    for campo in CAMPOS:
        celula = linha.select_one(f"td.{campo}")
        cliente[campo] = celula.get_text(strip=True) if celula else ""
    if all(cliente.values()):          # ignora linhas incompletas
        clientes.append(cliente)

# 3) Mostrar o resultado
print(f"Clientes encontrados: {len(clientes)}")
for cliente in clientes:
    print(cliente)

## 4. Envio ao N8N (o antigo `TODO`)

In [ ]:
if not N8N_WEBHOOK:
    print("N8N_WEBHOOK vazio: extração concluída, nada foi enviado (dry-run).")
else:
    resposta = requests.post(N8N_WEBHOOK, json={"clientes": clientes}, timeout=120)
    resposta.raise_for_status()
    dados = resposta.json()

    print("Resumo:", dados["resumo"])
    for m in dados["mensagens"]:
        print(f"\n--- {m['nome']} <{m['email']}> [{m['perfil']}] ---")
        print(m["mensagem"])
    for ignorado in dados["ignorados"]:
        print("\n[ignorado]", ignorado["nome"], "->", ", ".join(ignorado["motivos"]))